In [4]:
import numpy as np
import pandas as pd
import os

def sequential_split(interaction_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:

    train_parts, test_parts = [], []
    for u, g in interaction_df.groupby("user_id"):
        g = g.sort_values(["timestamp", "item_id"], kind="mergesort")
        if len(g) <= 1:
            continue

        test_parts.append(g.iloc[[-1]])
        train_parts.append(g.iloc[:-1])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df  = pd.concat(test_parts,  ignore_index=True)
    return train_df, test_df

raw_data_path = '../SHNS_data_raw'
output_data_path = '../SHNS_data'
dataset_names = ['Baby_5', 'Tools_and_Home_Improvement_5', 'Home_and_Kitchen_5', 'Cell_Phones_and_Accessories_5', 'Sports_and_Outdoors_5']
output_dir_names = ['baby_seq', 'tool_seq', 'home_seq', 'phone_seq', 'sports_seq']
for dataset_name, output_dir_name in zip(dataset_names, output_dir_names):

    dataset_path = raw_data_path + '/' + dataset_name + '.json'
    df = pd.read_json(dataset_path, lines=True)

    users = np.sort(df["reviewerID"].dropna().unique())
    items = np.sort(df["asin"].dropna().unique())
    user2id = {u:i for i, u in enumerate(users)}
    item2id = {a:i for i, a in enumerate(items)}

    user_ids = df["reviewerID"].map(user2id).astype("int64")
    item_ids = df["asin"].map(item2id).astype("int64")
    timestamps = df["unixReviewTime"].astype("int64")
    interaction_df = pd.DataFrame({"user_id": user_ids, "item_id": item_ids, "timestamp": timestamps})
    train_valid_df, test_df = sequential_split(interaction_df)
    train_df, valid_df = sequential_split(train_valid_df)

    cur_output_data_path = output_data_path + '/' + output_dir_name
    os.makedirs(cur_output_data_path, exist_ok=True)
    if os.path.exists(cur_output_data_path + "/train.txt"):
        print(f"{cur_output_data_path + '/train.txt'} already exists. Checking sanity...")
        existing_train_df = pd.read_csv(cur_output_data_path + "/train.txt", sep="\t", header=None, names=["user_id", "item_id"])
        assert train_df[["user_id", "item_id"]].equals(existing_train_df), "Existing train.txt does not match the newly created train_df!"
    else:
        train_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/train.txt", sep="\t", header=False, index=False)
    if os.path.exists(cur_output_data_path + "/valid.txt"):
        print(f"{cur_output_data_path + '/valid.txt'} already exists. Checking sanity...")
        existing_valid_df = pd.read_csv(cur_output_data_path + "/valid.txt", sep="\t", header=None, names=["user_id", "item_id"])
        assert valid_df[["user_id", "item_id"]].equals(existing_valid_df), "Existing valid.txt does not match the newly created valid_df!"
    else:
        valid_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/valid.txt", sep="\t", header=False, index=False)
    if os.path.exists(cur_output_data_path + "/test.txt"):
        print(f"{cur_output_data_path + '/test.txt'} already exists. Checking sanity...")
        existing_test_df = pd.read_csv(cur_output_data_path + "/test.txt", sep="\t", header=None, names=["user_id", "item_id"])
        assert test_df[["user_id", "item_id"]].equals(existing_test_df), "Existing test.txt does not match the newly created test_df!"
    else:
        test_df[["user_id", "item_id"]].to_csv(cur_output_data_path + "/test.txt",  sep="\t", header=False, index=False)


../SHNS_data/baby_seq/train.txt already exists. Checking sanity...
../SHNS_data/baby_seq/valid.txt already exists. Checking sanity...
../SHNS_data/baby_seq/test.txt already exists. Checking sanity...
../SHNS_data/tool_seq/train.txt already exists. Checking sanity...
../SHNS_data/tool_seq/valid.txt already exists. Checking sanity...
../SHNS_data/tool_seq/test.txt already exists. Checking sanity...
../SHNS_data/home_seq/train.txt already exists. Checking sanity...
../SHNS_data/home_seq/valid.txt already exists. Checking sanity...
../SHNS_data/home_seq/test.txt already exists. Checking sanity...
